In [1]:
# ── Google Drive folder ID (from the share link) ──────────────────────────────
DRIVE_FOLDER_ID = '1ktyrAnzQJxG-xOlgfJr70RpJQR2xwB4u'

# ── Where to write all clips (inside your Drive so results are persisted) ─────
# Will be created if it does not exist.
OUTPUT_CLIPS_DIR = '/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD_clips'

# ── Repo ──────────────────────────────────────────────────────────────────────
# For local development, use the current directory
# For Colab, use '/content/Driving_Distraction_Detection'
REPO_DIR = '/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection'
REPO_URL = 'https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git'

# ── Behaviour ─────────────────────────────────────────────────────────────────
SKIP_UNCLASSIFIED = False   # set True to drop 'unclassified' clips
DROP_KEYWORDS     = ('hand', 'head', 'face', 'gaze', 'eye', 'eyes', 'mirror')

print('Configuration loaded ✓')

Configuration loaded ✓


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')

# Authenticate so we can call the Drive API
auth.authenticate_user()
print('Drive mounted and authenticated ✓')

In [2]:
#Extract tar files 
import os
import tarfile
from pathlib import Path

dmd_root = "/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD"

# Check if folder exists
if os.path.isdir(dmd_root):
    print(f'✓ Folder "{dmd_root}" exists')
    
    # Find and extract all .tar files
    data_path = Path(dmd_root)
    tar_files = list(data_path.glob('**/*.tar'))
    
    if tar_files:
        print(f'Found {len(tar_files)} .tar files to extract...')
        for tar_path in tar_files:
            print(f'  Extracting {tar_path.name}...')
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=tar_path.parent)
        print('Extraction complete ✓')
    else:
        print('No .tar files found – proceeding with existing files.')
else:
    print(f'✗ Folder "{dmd_root}" does not exist')
    print('Please ensure the DMD folder is in the notebook directory')

✓ Folder "/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD" exists
Found 4 .tar files to extract...
  Extracting dmd-dataset-rgb_ir-gE-27.tar...
  Extracting dmd-dataset-rgb_ir-gE-30.tar...
  Extracting dmd-dataset-rgb_ir-gE-29.tar...
  Extracting dmd-dataset-rgb_ir-gE-28.tar...
Extraction complete ✓


In [4]:
%pip install -q opencv-python tqdm

import sys, os
repo_path = Path(REPO_DIR)
if repo_path.exists():
    print('Repo exists – pulling …')
    os.system(f'git -C "{REPO_DIR}" pull --ff-only')
else:
    print('Cloning repo …')
    os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')

scripts_path = str(repo_path / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os

# Google Drive setup
SCOPES = ['https://www.googleapis.com/auth/drive.file']
TOKEN_FILE = 'token.pickle'
CREDENTIALS_FILE = 'credentials.json'  # You'll need to download this from Google Cloud Console

def authenticate_drive():
    """Authenticate with Google Drive API"""
    creds = None
    
    # Load saved credentials
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, 'rb') as token:
            creds = pickle.load(token)
    
    # If no valid credentials, authenticate
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(CREDENTIALS_FILE):
                print(f"❌ Please download {CREDENTIALS_FILE} from Google Cloud Console")
                print("   Go to: https://console.cloud.google.com/")
                print("   Create a project → APIs & Services → Credentials → Create OAuth 2.0 Client ID")
                print("   Download the JSON file and save as 'credentials.json' in your project root")
                return None
                
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        
        # Save credentials
        with open(TOKEN_FILE, 'wb') as token:
            pickle.dump(creds, token)
    
    return build('drive', 'v3', credentials=creds)

def upload_to_drive(file_path, drive_folder_id=None, drive_service=None):
    """Upload a file to Google Drive"""
    if drive_service is None:
        drive_service = authenticate_drive()
        if drive_service is None:
            return None
    
    file_name = os.path.basename(file_path)
    
    # Create file metadata
    file_metadata = {'name': file_name}
    if drive_folder_id:
        file_metadata['parents'] = [drive_folder_id]
    
    # Upload file
    media = MediaFileUpload(file_path, resumable=True)
    file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id'
    ).execute()
    
    print(f"✅ Uploaded {file_name} to Google Drive (ID: {file.get('id')})")
    return file.get('id')

def upload_directory_to_drive(local_dir, drive_folder_id=None):
    """Upload entire directory to Google Drive"""
    drive_service = authenticate_drive()
    if drive_service is None:
        return
    
    print(f"Uploading contents of {local_dir} to Google Drive...")
    
    # Create main folder if drive_folder_id not provided
    if not drive_folder_id:
        folder_metadata = {
            'name': os.path.basename(local_dir),
            'mimeType': 'application/vnd.google-apps.folder'
        }
        folder = drive_service.files().create(
            body=folder_metadata,
            fields='id'
        ).execute()
        drive_folder_id = folder.get('id')
        print(f"Created folder: {os.path.basename(local_dir)} (ID: {drive_folder_id})")
    
    # Upload all files recursively
    for root, dirs, files in os.walk(local_dir):
        for file in files:
            file_path = os.path.join(root, file)
            upload_to_drive(file_path, drive_folder_id, drive_service)
    
    print(f"🎉 Upload complete! Files uploaded to: https://drive.google.com/drive/folders/{drive_folder_id}")
    return drive_folder_id

print('Google Drive upload functions ready ✓')

# ── Setup Instructions for Google Drive Upload ──────────────────────────────
"""
To upload files to Google Drive, you need to:

1. Go to Google Cloud Console: https://console.cloud.google.com/
2. Create a new project or select existing one
3. Enable the Google Drive API:
   - Go to "APIs & Services" → "Library"
   - Search for "Google Drive API" and enable it
4. Create OAuth 2.0 credentials:
   - Go to "APIs & Services" → "Credentials"
   - Click "Create Credentials" → "OAuth 2.0 Client IDs"
   - Choose "Desktop application"
   - Download the JSON file and save it as 'credentials.json' in your project root
5. Run the upload function:
   - upload_directory_to_drive(str(clips_dir))
   - This will open a browser for authentication on first run
"""

You should consider upgrading via the '/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Repo exists – pulling …
Already up to date.
Google Drive upload functions ready ✓


'\nTo upload files to Google Drive, you need to:\n\n1. Go to Google Cloud Console: https://console.cloud.google.com/\n2. Create a new project or select existing one\n3. Enable the Google Drive API:\n   - Go to "APIs & Services" → "Library"\n   - Search for "Google Drive API" and enable it\n4. Create OAuth 2.0 credentials:\n   - Go to "APIs & Services" → "Credentials"\n   - Click "Create Credentials" → "OAuth 2.0 Client IDs"\n   - Choose "Desktop application"\n   - Download the JSON file and save it as \'credentials.json\' in your project root\n5. Run the upload function:\n   - upload_directory_to_drive(str(clips_dir))\n   - This will open a browser for authentication on first run\n'

In [5]:
from __future__ import annotations

import json, re
from typing import Any

import cv2
from tqdm.notebook import tqdm

from clean_dmd_body_actions import clean_dmd_json  # from cloned repo

DMD_BODY_ACTION_MAP: dict[str, int] = {
    'safe_driving':         0,
    'texting_right':        1,
    'phonecall_right':      2,
    'texting_left':         3,
    'phonecall_left':       4,
    'radio':                5,
    'drinking':             6,
    'reach_side':           7,
    'hair_and_makeup':      8,
    'talking_to_passenger': 9,
    'reach_backseat':       10,
    'change_gear':          11,
    'stand_still_waiting':  12,
    'unclassified':         13,
}

# Mapping from action type in JSON to class name
type_to_class: dict[str, str] = {}
for cls in DMD_BODY_ACTION_MAP:
    if cls == 'safe_driving':
        type_to_class['driver_actions/safe_drive'] = cls
    elif cls == 'talking_to_passenger':
        type_to_class['talking/talking'] = cls
    else:
        type_to_class[f'driver_actions/{cls}'] = cls

print(f'Imports OK ✓  ({len(DMD_BODY_ACTION_MAP)} classes)')

Imports OK ✓  (14 classes)


In [6]:
_VIDEO_STEM_RE = re.compile(r'^(.+)_rgb_body$')


def find_leaf_folders(root: Path) -> list[Path]:
    """
    Recursively walk *root* and return every folder that contains at least
    one '*_rgb_body.mp4' file (= a 'base case' ready for processing).
    """
    leaves: list[Path] = []
    for folder in sorted(root.rglob('*')):
        if not folder.is_dir():
            continue
        # A leaf folder contains at least one body-camera video
        videos = [f for f in folder.iterdir()
                  if f.suffix == '.mp4' and _VIDEO_STEM_RE.match(f.stem)]
        if videos:
            leaves.append(folder)
    return leaves


def find_pairs_in_folder(folder: Path) -> list[tuple[Path, Path]]:
    """
    Inside a leaf folder, pair each '*_rgb_body.mp4' with its
    '*_rgb_ann_distraction_body_only.json' (already cleaned).
    """
    pairs: list[tuple[Path, Path]] = []
    for mp4 in sorted(folder.glob('*.mp4')):
        m = _VIDEO_STEM_RE.match(mp4.stem)
        if not m:
            continue
        prefix    = m.group(1)
        json_path = folder / f'{prefix}_rgb_ann_distraction_body_only.json'
        if json_path.exists():
            pairs.append((mp4, json_path))
        else:
            print(f'  [WARN] No body_only JSON for {mp4.name}')
    return pairs


def load_actions(json_path: Path) -> list[dict[str, Any]]:
    with json_path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    actions_raw = payload.get('openlabel', {}).get('actions', {})
    actions: list[dict[str, Any]] = []
    for action_id, action_data in actions_raw.items():
        atype = str(action_data.get('type', 'unclassified')).lower()
        for iv in action_data.get('frame_intervals', []):
            actions.append({
                'id': action_id, 'type': atype,
                'frame_start': int(iv['frame_start']),
                'frame_end':   int(iv['frame_end']),
            })
    actions.sort(key=lambda a: (a['frame_start'], a['frame_end']))
    return actions


def write_clip(cap, out_path: Path, fs: int, fe: int,
               fps: float, w: int, h: int) -> bool:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, fs)
    written = 0
    for _ in range(fe - fs + 1):
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        written += 1
    writer.release()
    if written == 0:
        out_path.unlink(missing_ok=True)
        return False
    return True


def sanitize(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', s)


print('Helpers defined ✓')

Helpers defined ✓


In [17]:
%pip install ipywidgets

     |████████████████████████████████| 139 kB 8.8 MB/s eta 0:00:01
     |████████████████████████████████| 2.2 MB 55.2 MB/s eta 0:00:01
     |████████████████████████████████| 914 kB 105.3 MB/s eta 0:00:01
You should consider upgrading via the '/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
DATA_ROOT = Path('/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD/dmd')
clips_dir = Path(OUTPUT_CLIPS_DIR)
clips_dir.mkdir(parents=True, exist_ok=True)
for cls in DMD_BODY_ACTION_MAP:
    (clips_dir / cls).mkdir(exist_ok=True)

print(f'Output dir: {clips_dir}')
print('Class subfolders created ✓')

# Find all leaf folders
leaf_folders = find_leaf_folders(DATA_ROOT)
print(f'\nLeaf folders found: {len(leaf_folders)}')
for lf in leaf_folders:
    print(f'  {lf.relative_to(DATA_ROOT)}')

drop_kws    = tuple(k.lower() for k in DROP_KEYWORDS)
label_lines: list[str]  = []
stats: dict[str, int]   = {k: 0 for k in DMD_BODY_ACTION_MAP}

for leaf in tqdm(leaf_folders, desc='Leaf folders', unit='folder'):
    print(f'\n📂 {leaf.relative_to(DATA_ROOT)}')

    # ── Step A: clean raw annotation JSONs in this folder ────────────────────
    raw_jsons = [
        p for p in leaf.glob('*.json')
        if 'ann_distraction' in p.stem and '_body_only' not in p.stem
    ]
    for raw_json in raw_jsons:
        out_json = raw_json.with_name(f'{raw_json.stem}_body_only.json')
        if out_json.exists():
            print(f'  [SKIP clean] {out_json.name}')
            continue
        result   = clean_dmd_json(raw_json, out_json, drop_kws)
        n_act    = len(result.get('openlabel', {}).get('actions', {}))
        print(f'  ✓ cleaned {out_json.name}  ({n_act} body actions)')

    # ── Step B: find video–annotation pairs ──────────────────────────────────
    pairs = find_pairs_in_folder(leaf)
    if not pairs:
        print('  [WARN] No video–annotation pairs found, skipping.')
        continue

    # ── Step C: cut each video ───────────────────────────────────────────────
    for video_path, json_path in pairs:
        actions = load_actions(json_path)
        if not actions:
            print(f'  [WARN] No actions in {json_path.name}')
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f'  [ERROR] Cannot open {video_path.name}')
            continue

        fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'  ✂  {video_path.name}  [{total} frames | {fps:.1f} fps | {w}×{h}]')

        for idx, action in enumerate(
            tqdm(actions, desc='    clips', unit='clip', leave=False)
        ):
            label_str = type_to_class.get(action['type'], 'unclassified')
            class_id  = DMD_BODY_ACTION_MAP.get(label_str, 13)  # 13 = unclassified

            if SKIP_UNCLASSIFIED and class_id == 13:
                continue

            fs = max(0, action['frame_start'])
            fe = min(total - 1, action['frame_end'])
            if fs > fe:
                continue

            clip_name = f'{video_path.stem}_clip{idx:04d}_{sanitize(label_str)}.mp4'
            out_path  = clips_dir / label_str / clip_name

            if out_path.exists():
                # Re-running: skip write but still register label
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1
                continue

            if write_clip(cap, out_path, fs, fe, fps, w, h):
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1

        cap.release()

print(f'\n✅ Done — {len(label_lines)} clips total')

Output dir: /Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD_clips
Class subfolders created ✓

Leaf folders found: 11
  gE/27/s1
  gE/27/s2
  gE/28/s1
  gE/28/s2
  gE/28/s4
  gE/29/s1
  gE/29/s2
  gE/29/s3
  gE/30/s1
  gE/30/s2
  gE/30/s3


Leaf folders:   0%|          | 0/11 [00:00<?, ?folder/s]


📂 gE/27/s1
  ✓ cleaned gE_27_s1_2019-03-07T13;14;28+01;00_rgb_ann_distraction_body_only.json  (2 body actions)
  ✓ cleaned gE_27_s1_2019-03-07T13;18;37+01;00_rgb_ann_distraction_body_only.json  (7 body actions)
  ✂  gE_27_s1_2019-03-07T13;14;28+01;00_rgb_body.mp4  [7089 frames | 29.8 fps | 1280×720]


    clips:   0%|          | 0/47 [00:00<?, ?clip/s]

  ✂  gE_27_s1_2019-03-07T13;18;37+01;00_rgb_body.mp4  [12208 frames | 29.8 fps | 1280×720]


...


📂 gE/30/s2
  ✓ cleaned gE_30_s2_2019-03-15T10;43;40+01;00_rgb_ann_distraction_body_only.json  (10 body actions)
  ✂  gE_30_s2_2019-03-15T10;43;40+01;00_rgb_body.mp4  [13400 frames | 29.8 fps | 1280×720]


    clips:   0%|          | 0/85 [00:00<?, ?clip/s]


📂 gE/30/s3
  ✓ cleaned gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction_body_only.json  (5 body actions)
  ✂  gE_30_s3_2019-03-15T10;56;02+01;00_rgb_body.mp4  [1863 frames | 29.8 fps | 1280×720]


    clips:   0%|          | 0/12 [00:00<?, ?clip/s]


✅ Done — 1018 clips total


In [ ]:
labels_path = clips_dir / 'labels.txt'
with labels_path.open('w', encoding='utf-8') as fh:
    fh.write('\n'.join(label_lines))
    if label_lines:
        fh.write('\n')

print(f'Written: {labels_path}  ({len(label_lines)} lines)')
print('\nSample (first 10 lines):')
for line in label_lines[:10]:
    print(' ', line)

In [ ]:
print(f"{'Class':<30} {'ID':>4}  {'Clips':>6}")
print('-' * 50)
total = 0
for cls, cid in DMD_BODY_ACTION_MAP.items():
    count  = stats.get(cls, 0)
    total += count
    bar    = '█' * min(count, 40)
    print(f'{cls:<30} {cid:>4}  {count:>6}  {bar}')
print('-' * 50)
print(f"{'TOTAL':<30}       {total:>6}")

In [ ]:
# ── Upload to Shared DMD_Clips Folder ───────────────────────────────────────
# To upload to your shared "DMD_clips" folder in Google Drive Shareddrives:
# 1. Open the shared folder in Google Drive: https://drive.google.com/drive/folders/YOUR_FOLDER_ID
# 2. Copy the folder ID from the URL (the long string after /folders/)
# 3. Replace 'YOUR_SHARED_DMD_CLIPS_FOLDER_ID' below with the actual ID

SHARED_DMD_CLIPS_FOLDER_ID = '1OdgFEcbz8fr5CS2WZxNqyj8M4JeTbXcJ'  # ← Your shared DMD_clips folder ID

if SHARED_DMD_CLIPS_FOLDER_ID != 'YOUR_SHARED_DMD_CLIPS_FOLDER_ID':
    print(f"📤 Uploading processed clips to shared DMD_clips folder...")
    upload_directory_to_drive(str(clips_dir), SHARED_DMD_CLIPS_FOLDER_ID)
    print("✅ Upload complete!")
    print(f"🔗 Access your files at: https://drive.google.com/drive/folders/{SHARED_DMD_CLIPS_FOLDER_ID}")
else:
    print("⚠️  To upload to your shared DMD_clips folder:")
    print("   1. Open the shared DMD_clips folder in Google Drive")
    print("   2. Copy the folder ID from the URL (after /folders/)")
    print("   3. Replace 'YOUR_SHARED_DMD_CLIPS_FOLDER_ID' above")
    print("   4. Re-run this cell")
    print()
    print("💡 Alternative: Run this to create a new folder:")
    print("   upload_directory_to_drive(str(clips_dir))  # Creates 'DMD_clips' folder")